# Control: System Modeling and Controllers

**Learning Goals**
- Understand how to model a physical system for control design
- See how open-loop control fails for the block on ice
- See how closed-loop (proportional) control can be beneficial
- Adjust controller gains and observe the effect on system response

## System Modeling

Before we can control a system, we need a **model** of that system that relates inputs to outputs.

For the **block on ice** introduced in the systems notebook, Newton's second law gives:

$$m \ddot x = F - b \dot x$$

where $m$ is mass, $b$ is the friction coefficient, $F$ is the applied force (input), and $x$ is the position (output).

We will use $m = 10$ kg and $b = 0.5$ N·s/m.

---

In [1]:
%pip install -q ipywidgets

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from scipy.integrate import solve_ivp
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets

print("Libraries loaded.")

Libraries loaded.


In [3]:
def simulate_block(control_func, t_span=30.0, dt=1/240, friction=0.5, mass=10.0):
    def ode(t, state):
        x, v = state
        force = control_func(t, x, v)
        friction_force = -friction * v
        dxdt = v
        dvdt = (force + friction_force) / mass
        return [dxdt, dvdt]

    t_eval = np.arange(0, t_span, dt)
    sol = solve_ivp(ode, [0, t_span], [0.0, 0.0], t_eval=t_eval, method='RK45')
    forces = np.array([control_func(t, sol.y[0, i], sol.y[1, i]) for i, t in enumerate(sol.t)])
    return sol.t, sol.y[0], forces

def make_animation(times, positions, forces, title, arrow_scale=0.08, color='b'):
    n_frames = 30
    step = max(1, len(times) // n_frames)
    t = times[::step][:n_frames]
    y = positions[::step][:n_frames]
    f = forces[::step][:n_frames]
    x_max = max(25, np.ceil(max(positions) + 2))
    f_max = max(abs(forces)) if len(forces) else 1

    fig = plt.figure(figsize=(8, 5.5))
    ax1 = fig.add_subplot(2, 1, 1)
    ax1.set_xlim(-1, x_max)
    ax1.set_ylim(-0.5, 3.5)
    ax1.set_aspect('equal')
    ax1.axhline(0, color='gray', linewidth=2)
    ax1.axvline(10, color='k', ls='--', alpha=0.4, linewidth=1.5, label='Setpoint = 10 m')
    ax1.set_ylabel('Block')
    ax1.legend(loc='upper right', fontsize=9)
    ax1.set_title(title, fontsize=11)
    block = Rectangle((0, 0), 2, 1, fc=color, ec='k', lw=2)
    ax1.add_patch(block)
    time_text = ax1.text(0.02, 0.88, '', transform=ax1.transAxes, fontsize=9)
    force_text = ax1.text(0.02, 0.78, '', transform=ax1.transAxes, fontsize=9, color=color)

    ax2 = fig.add_subplot(2, 1, 2)
    ax2.set_xlim(0, 30)
    ax2.set_ylim(-1, x_max)
    ax2.axhline(10, color='k', ls='--', alpha=0.4)
    ax2.set(xlabel='Time (s)', ylabel='Position (m)')
    ax2.grid(alpha=0.3)
    line, = ax2.plot([], [], color=color, linewidth=2)

    fig.tight_layout()

    def init():
        block.set_xy((-1, 0))
        line.set_data([], [])
        time_text.set_text('')
        force_text.set_text('')
        return block, line, time_text, force_text

    def animate(i):
        xi = y[i]
        fi = f[i]
        block.set_xy((xi - 1, 0))
        line.set_data(t[:i], y[:i])
        time_text.set_text(f't = {t[i]:.1f} s')
        force_text.set_text(f'F = {fi:+.1f} N')
        return block, line, time_text, force_text

    ani = FuncAnimation(fig, animate, frames=len(t),
                        init_func=init, blit=True, interval=100)
    plt.close(fig)
    return HTML(ani.to_jshtml())

print("Helpers defined.")

Helpers defined.


---

## Open-Loop Control

In **open-loop control**, the controller decides the force based only on the desired setpoint. It does **not** use any information about the block's actual position.

A simple open-loop strategy: apply a constant force $F = K \cdot r$, where $r = 10$ m is the setpoint and $K$ is a **gain** we can adjust.

**Click Run to see it in action!**

In [4]:
K_slider = widgets.FloatSlider(min=0.0, max=3.0, step=0.05, value=1.0,
                                description='Gain K:', style={'description_width': 'initial'})
run_btn = widgets.Button(description='Run', button_style='primary')
ol_out = widgets.Output()

def on_run_ol(b):
    K = round(K_slider.value, 2)
    ctrl = lambda t, x, v: K * 10.0
    times, positions, forces = simulate_block(ctrl)
    with ol_out:
        clear_output(wait=True)
        display(make_animation(times, positions, forces,
                               f'Open-Loop K={K:.2f}', color='b'))

run_btn.on_click(on_run_ol)
display(widgets.VBox([K_slider, run_btn, ol_out]))

**Observation.** With open-loop control the block accelerates until the ice friction matches the applied force: it reaches a constant terminal velocity. The block never settles at the setpoint because the controller doesn't know where the block actually is.

---

## Closed-Loop (Proportional) Control

In **closed-loop control**, the controller uses the current position to decide the force. The simplest closed-loop strategy is **proportional (P) control**:

$$F(t) = K_p \, \bigl(r - x(t)\bigr)$$

Here $e(t) = r - x(t)$ is the **error**, the difference between where we want the block and where it actually is.

Ice friction is included in the physics (proportional to velocity), so the block will experience some **damping**. With P-control and damping, the block can settle at the setpoint, but it will **oscillate** on the way there if the damping is light.

**Click Run!**

In [5]:
Kp_slider = widgets.FloatSlider(min=0.1, max=5.0, step=0.1, value=1.0,
                                 description='Kp:', style={'description_width': 'initial'})
run_btn2 = widgets.Button(description='Run', button_style='primary')
pc_out = widgets.Output()

def on_run_pc(b):
    Kp = round(Kp_slider.value, 1)
    ctrl = lambda t, x, v: Kp * (10.0 - x)
    times, positions, forces = simulate_block(ctrl)
    with pc_out:
        clear_output(wait=True)
        display(make_animation(times, positions, forces,
                               f'P-Control Kp={Kp:.1f}', color='r'))

run_btn2.on_click(on_run_pc)
display(widgets.VBox([Kp_slider, run_btn2, pc_out]))

**Observation.** The proportional controller pulls the block toward the setpoint.

With ice friction providing light damping, the block exhibits **underdamped** response: it overshoots, swings back, overshoots less, and eventually settles at the setpoint. Lower $K_p$ gives a sluggish response with little overshoot; higher $K_p$ makes it faster but more oscillatory.